# 04 Tool Calling：单次调用与多轮工具循环

演示如何声明工具、解析模型返回的工具调用、执行本地函数，并将结果回传给模型完成多轮循环。

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("HY3_API_KEY"),
    base_url="https://tokenhub.tencentmaas.com/v1",
)

def get_weather(city: str) -> str:
    weather_db = {"北京": "晴朗，25°C", "上海": "多云，28°C", "深圳": "小雨，30°C"}
    return weather_db.get(city, f"暂无 {city} 的天气数据")

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"}
                },
                "required": ["city"],
            },
        },
    }
]

messages = [
    {"role": "system", "content": "你可以调用 get_weather 工具查询天气。"},
    {"role": "user", "content": "今天北京和深圳的天气怎么样？"},
]

response = client.chat.completions.create(
    model="hy3",
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0.3,
)

message = response.choices[0].message
print(message)

In [ ]:
messages.append(message)

for tool_call in message.tool_calls:
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    print(f"调用工具: {function_name}({function_args})")
    result = get_weather(**function_args)
    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

final_response = client.chat.completions.create(
    model="hy3",
    messages=messages,
    tools=tools,
    temperature=0.3,
)

print(final_response.choices[0].message.content)